# Onset & Tempo Detection — Ballroom Dataset

**Project 2 | Audio Technology II**

This notebook implements and evaluates two onset detection systems:
1. **Baseline**: onset-strength envelope + fixed global threshold
2. **Improved**: RMS energy novelty function + local-maxima peak picking
3. **Extension**: tempo estimation from inter-onset intervals (IOI)

Both systems are evaluated against ground-truth beat annotations from the Ballroom dataset using precision, recall, and F-measure.

## 0. Imports

In [ ]:
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import find_peaks
from collections import Counter
from IPython.display import Audio

## 1. Paths — update these to your local Ballroom folder

Expected structure:
```
BallroomData/
  ChaChaCha/
    Albums-Cafe_Paradiso-05.wav
    ...
  Waltz/
    ...
Ballroom CSVs/
  beat_csv/
    Albums-Cafe_Paradiso-05_beats.csv
    ...
  tempo_groundtruth.csv
```

In [ ]:
# ── UPDATE THESE PATHS ────────────────────────────────────────────────────
AUDIO_ROOT = "/path/to/BallroomData"
BEAT_CSV_DIR = "/path/to/Ballroom CSVs/beat_csv"
TEMPO_CSV = "/path/to/Ballroom CSVs/tempo_groundtruth.csv"
# ─────────────────────────────────────────────────────────────────────────

# Single test file used for demos in Sections 2–5
DEMO_AUDIO = AUDIO_ROOT + "/ChaChaCha/Albums-Cafe_Paradiso-05.wav"
DEMO_BEATS = BEAT_CSV_DIR + "/Albums-Cafe_Paradiso-05_beats.csv"

## 2. Load a Demo File

In [ ]:
# Load audio at 22050 Hz (downsampled from 44100 to reduce compute time)
x, sr = librosa.load(DEMO_AUDIO, sr=22050, mono=True)

# Load ground-truth beat times from the matching CSV
beats_df = pd.read_csv(DEMO_BEATS)
click_times = beats_df["beat_time"].values

print(f"Sample rate : {sr} Hz")
print(f"Duration    : {len(x)/sr:.1f} s")
print(f"GT beats    : {len(click_times)}")

Audio(x, rate=sr)

## 3. Baseline Onset Detector

**Approach:**
- Compute `librosa.onset.onset_strength` (spectral flux under the hood)
- Normalize to [0, 1]
- Pick peaks above a fixed global threshold using `scipy.signal.find_peaks`

This is deliberately simple — it gives us a reference point to beat with the improved system.

In [ ]:
def baseline_onset_detector(x, sr, hop_length=512, peak_height=0.3):
    """
    Detect onsets using librosa's onset-strength envelope and a fixed threshold.

    Parameters
    ----------
    x           : np.ndarray  Mono audio signal.
    sr          : int         Sample rate (Hz).
    hop_length  : int         Number of samples between successive frames.
                              Controls time resolution of the onset envelope.
                              Default 512 → ~23 ms steps at 22050 Hz.
    peak_height : float       Minimum normalized onset-strength value a frame
                              must reach to be counted as an onset (0–1 scale).
                              Default 0.3 means only the top 70% of the envelope
                              is considered. A higher value → fewer detections
                              (higher precision, lower recall).

    Returns
    -------
    onset_times : np.ndarray  Detected onset times in seconds.
    onset_env   : np.ndarray  Normalized onset-strength envelope (for plotting).

    How it works
    ------------
    1. onset_strength computes spectral flux — the sum of positive differences
       between successive magnitude spectra. Sudden spectral changes (attacks)
       produce large values.
    2. Normalizing to [0, 1] makes the threshold comparable across files with
       different loudness levels.
    3. find_peaks with a global height cutoff picks every frame above the
       threshold. `distance=2` prevents two detections within 2 frames (~46 ms)
       but is otherwise quite permissive.
    """

    # Step 1: Compute onset-strength envelope (spectral flux)
    onset_env = librosa.onset.onset_strength(y=x, sr=sr, hop_length=hop_length)

    # Step 2: Normalize to [0, 1] so the threshold is scale-independent
    onset_env = onset_env / np.max(onset_env)

    # Step 3: Pick peaks above the global threshold.
    # distance=2 prevents detections within 2 consecutive frames.
    peaks, _ = find_peaks(onset_env, height=peak_height, distance=2)

    # Step 4: Convert frame indices to seconds
    onset_times = librosa.frames_to_time(peaks, sr=sr, hop_length=hop_length)

    # Time axis for plotting
    t = librosa.frames_to_time(np.arange(len(onset_env)), sr=sr, hop_length=hop_length)

    return onset_times, onset_env, t

In [ ]:
onset_times_base, onset_env, t_env = baseline_onset_detector(x, sr)

# ── Plot: waveform + GT beats + detected onsets ───────────────────────────
clip_s = 10   # show first 10 seconds
t_audio = np.arange(len(x)) / sr

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

# Top panel: waveform with beat markers
axes[0].plot(t_audio[:clip_s * sr], x[:clip_s * sr], alpha=0.4, color='steelblue')
for t in click_times[click_times <= clip_s]:
    axes[0].axvline(t, color='green', lw=1.4, alpha=0.8)
for t in onset_times_base[onset_times_base <= clip_s]:
    axes[0].axvline(t, color='red', lw=0.9, alpha=0.5, linestyle='--')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Baseline — Waveform with Ground Truth (green) vs Detected (red)')
gt_patch  = mpatches.Patch(color='green', label='Ground truth')
det_patch = mpatches.Patch(color='red',   label='Detected')
axes[0].legend(handles=[gt_patch, det_patch])

# Bottom panel: onset-strength envelope + threshold line
mask = t_env <= clip_s
axes[1].plot(t_env[mask], onset_env[mask], color='orange', label='Onset strength')
axes[1].axhline(0.3, color='grey', linestyle=':', label='Threshold = 0.3')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Onset Strength')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"GT beats     : {len(click_times)}")
print(f"Detected     : {len(onset_times_base)}")

## 4. Improved Onset Detector

**Improvements over baseline:**

| | Baseline | Improved |
|---|---|---|
| Feature | Spectral flux (`onset_strength`) | RMS energy |
| Novelty | Raw envelope | Discrete derivative + half-wave rectification |
| Peak picking | Global fixed threshold | Local-maxima check + threshold |

**Why RMS + derivative?**  
RMS is a smoother measure of signal strength. Taking its frame-to-frame difference (∆E) highlights sudden energy *increases* — exactly what an onset looks like. Half-wave rectification (setting negatives to 0) discards energy *decreases*, which are irrelevant for onset detection.

**Why local maxima instead of global threshold?**  
A peak that simply clears the threshold on a broad plateau will fire multiple times. Requiring a frame to beat *both* its immediate neighbors ensures we only pick the sharpest point of a transient.

In [ ]:
def improved_onset_detector(x, sr, hop_length=512, frame_length=1024, threshold=0.15):
    """
    Detect onsets using an RMS-based novelty function and local-maxima peak picking.

    Parameters
    ----------
    x            : np.ndarray  Mono audio signal.
    sr           : int         Sample rate (Hz).
    hop_length   : int         Hop size in samples. Controls time resolution.
                               Default 512 → ~23 ms steps at 22050 Hz.
    frame_length : int         Window size for RMS computation (samples).
                               Larger values → smoother envelope.
                               Default 1024 → ~46 ms window at 22050 Hz.
    threshold    : float       Minimum novelty value (after normalization to [0,1])
                               a local maximum must exceed to be kept as an onset.
                               Default 0.15. Increase to reduce false positives.

    Returns
    -------
    onset_times : np.ndarray  Detected onset times in seconds.
    novelty     : np.ndarray  Normalized novelty function (for plotting/evaluation).
    t           : np.ndarray  Time axis matching novelty.

    How it works
    ------------
    Step 1 — RMS energy:
        Compute the root-mean-square energy per frame.
        RMS = sqrt(mean(x^2)) over each window.
        This gives a smooth measure of signal loudness that is less
        sensitive to noise than raw energy.

    Step 2 — Discrete derivative (novelty function):
        ∆E(n) = RMS(n+1) - RMS(n)
        This highlights *changes* in energy level. A sudden increase
        (attack/onset) produces a large positive value.

    Step 3 — Half-wave rectification:
        Set all negative values to 0. Onsets are energy increases,
        so we discard decreases (note endings, decay).

    Step 4 — Normalize to [0, 1]:
        Makes the threshold consistent across files with different
        loudness levels.

    Step 5 — Local-maxima peak picking:
        A frame is an onset only if:
          (a) its novelty value exceeds `threshold`, AND
          (b) it is greater than both its immediate neighbours.
        This prevents flat plateaus or the shoulders of broad peaks
        from firing multiple times.
    """

    # Step 1: RMS energy per frame
    rmse = librosa.feature.rms(
        y=x,
        frame_length=frame_length,
        hop_length=hop_length,
        center=True
    )[0]   # shape: (n_frames,)

    # Step 2: Discrete derivative — ΔE(n) = RMS(n+1) − RMS(n)
    rmse_diff = np.diff(rmse)
    # Pad with a trailing 0 so the array stays the same length as rmse
    novelty = np.concatenate([rmse_diff, [0]])

    # Step 3: Half-wave rectification — keep only positive energy changes
    novelty[novelty < 0] = 0

    # Step 4: Normalize to [0, 1]
    if novelty.max() > 0:
        novelty = novelty / novelty.max()

    # Time axis: convert frame indices to seconds
    t = librosa.frames_to_time(np.arange(len(novelty)), sr=sr, hop_length=hop_length)

    # Step 5: Local-maxima peak picking with threshold
    peak_indices = []
    for i in range(1, len(novelty) - 1):
        # Must exceed threshold AND beat both neighbours
        if novelty[i] > threshold:
            if novelty[i] > novelty[i - 1] and novelty[i] > novelty[i + 1]:
                peak_indices.append(i)

    peak_indices = np.array(peak_indices)
    onset_times  = t[peak_indices] if len(peak_indices) > 0 else np.array([])

    return onset_times, novelty, t

In [ ]:
onset_times_imp, novelty, t_nov = improved_onset_detector(x, sr)

# ── Plot: waveform + GT beats + detected onsets ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(t_audio[:clip_s * sr], x[:clip_s * sr], alpha=0.4, color='steelblue')
for t in click_times[click_times <= clip_s]:
    axes[0].axvline(t, color='green', lw=1.4, alpha=0.8)
for t in onset_times_imp[onset_times_imp <= clip_s]:
    axes[0].axvline(t, color='crimson', lw=0.9, alpha=0.5, linestyle='--')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Improved — Waveform with Ground Truth (green) vs Detected (red)')
axes[0].legend(handles=[gt_patch, det_patch])

mask = t_nov <= clip_s
axes[1].plot(t_nov[mask], novelty[mask], color='orange', label='Novelty (RMS derivative)')
axes[1].axhline(0.15, color='grey', linestyle=':', label='Threshold = 0.15')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Novelty')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"GT beats     : {len(click_times)}")
print(f"Detected     : {len(onset_times_imp)}")

## 5. Evaluation Function

We use **precision**, **recall**, and **F-measure** with a time-tolerance window (default 50 ms).  
Each detected onset is matched to at most one ground-truth beat — greedy, sorted by time.

In [ ]:
def evaluate_onsets(predicted, ground_truth, tolerance=0.05):
    """
    Evaluate onset detection against ground-truth beat times.

    Parameters
    ----------
    predicted    : array-like  Detected onset times in seconds.
    ground_truth : array-like  Ground-truth beat times in seconds.
    tolerance    : float       Maximum allowed timing error in seconds.
                               Default 0.05 s = 50 ms. A predicted onset is
                               considered correct if it lands within this
                               window around any ground-truth beat.

    Returns
    -------
    precision : float  Fraction of detected onsets that matched a GT beat.
                       High precision → few false positives.
    recall    : float  Fraction of GT beats that were detected.
                       High recall    → few missed beats.
    f_measure : float  Harmonic mean of precision and recall.
                       Single summary score: 1.0 = perfect, 0.0 = no matches.

    How it works
    ------------
    Greedy matching: for each predicted onset (in order), find the first
    unmatched ground-truth beat within the tolerance window. Each GT beat
    can only be claimed once.
    """
    matches, used = 0, set()

    for p in predicted:
        for i, gt in enumerate(ground_truth):
            if i not in used and abs(p - gt) <= tolerance:
                matches += 1
                used.add(i)  # mark this GT beat as claimed
                break

    precision = matches / len(predicted)    if len(predicted)    > 0 else 0.0
    recall    = matches / len(ground_truth) if len(ground_truth) > 0 else 0.0
    f = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f

## 6. Naïve Baseline — Random Chance

To put the results in context we compare against a **random detector** that fires onsets at uniformly random times.  
We match the number of detections to the improved system to keep the comparison fair.

In [ ]:
def random_onset_detector(duration, n_onsets, seed=42):
    """
    Generate random onset times as a naïve baseline.

    Parameters
    ----------
    duration  : float  Length of the audio in seconds.
    n_onsets  : int    Number of onsets to generate (matched to the improved
                       detector's count so the comparison is fair).
    seed      : int    Random seed for reproducibility.

    Returns
    -------
    onset_times : np.ndarray  Sorted random onset times in seconds.

    Why include this?
    -----------------
    A model that achieves, say, 0.60 F-measure is only useful if random
    guessing performs much worse. This sanity-check confirms our detectors
    are actually learning something from the signal.
    """
    rng = np.random.default_rng(seed)
    return np.sort(rng.uniform(0, duration, n_onsets))


duration_s = len(x) / sr
rand_onsets = random_onset_detector(duration_s, n_onsets=len(onset_times_imp))
p_r, r_r, f_r = evaluate_onsets(rand_onsets, click_times)

## 7. Single-File Evaluation — Baseline vs Improved vs Random

In [ ]:
p_b, r_b, f_b = evaluate_onsets(onset_times_base, click_times)
p_i, r_i, f_i = evaluate_onsets(onset_times_imp,  click_times)

print(f"{'Model':<12} {'Precision':>10} {'Recall':>10} {'F-measure':>10}")
print("-" * 45)
print(f"{'Baseline':<12} {p_b:>10.3f} {r_b:>10.3f} {f_b:>10.3f}")
print(f"{'Improved':<12} {p_i:>10.3f} {r_i:>10.3f} {f_i:>10.3f}")
print(f"{'Random':<12} {p_r:>10.3f} {r_r:>10.3f} {f_r:>10.3f}")

# ── Bar chart: single-file comparison ────────────────────────────────────
labels  = ['Precision', 'Recall', 'F-measure']
x_pos   = np.arange(len(labels))
width   = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_pos - width, [p_b, r_b, f_b], width, label='Baseline',  color='steelblue')
ax.bar(x_pos,         [p_i, r_i, f_i], width, label='Improved',  color='crimson')
ax.bar(x_pos + width, [p_r, r_r, f_r], width, label='Random',    color='grey', alpha=0.6)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Onset Detection — Single File Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Full-Dataset Evaluation

Run both detectors over every file in the Ballroom dataset and collect results per file.  
**Note:** this cell can take several minutes depending on how many files you have.

In [ ]:
import os

def run_dataset_evaluation(audio_root, beat_csv_dir, sr_target=22050):
    """
    Run baseline, improved, and random detectors over all WAV files in
    `audio_root` and return a DataFrame of per-file evaluation metrics.

    Parameters
    ----------
    audio_root   : str    Root folder containing genre sub-folders of WAV files.
    beat_csv_dir : str    Folder containing per-file beat CSVs named
                          `<basename>_beats.csv`.
    sr_target    : int    Sample rate to load audio at. Default 22050 Hz.

    Returns
    -------
    results : pd.DataFrame  One row per file with columns:
              file, style, p_base, r_base, f_base,
              p_impr, r_impr, f_impr, p_rand, r_rand, f_rand.
    """
    rows = []

    for style in sorted(os.listdir(audio_root)):
        style_path = os.path.join(audio_root, style)
        if not os.path.isdir(style_path):
            continue

        for fname in sorted(os.listdir(style_path)):
            if not fname.endswith('.wav'):
                continue

            wav_path  = os.path.join(style_path, fname)
            base      = fname.replace('.wav', '')
            beat_path = os.path.join(beat_csv_dir, base + '_beats.csv')

            if not os.path.exists(beat_path):
                continue   # skip if no matching annotation

            try:
                y, sr  = librosa.load(wav_path, sr=sr_target, mono=True)
                gt     = pd.read_csv(beat_path)['beat_time'].values

                # Run all three detectors
                ot_b, _, _ = baseline_onset_detector(y, sr)
                ot_i, _, _ = improved_onset_detector(y, sr)
                ot_r       = random_onset_detector(len(y)/sr, n_onsets=len(ot_i))

                p_b, r_b, f_b = evaluate_onsets(ot_b, gt)
                p_i, r_i, f_i = evaluate_onsets(ot_i, gt)
                p_r, r_r, f_r = evaluate_onsets(ot_r, gt)

                rows.append(dict(
                    file=fname, style=style,
                    p_base=p_b, r_base=r_b, f_base=f_b,
                    p_impr=p_i, r_impr=r_i, f_impr=f_i,
                    p_rand=p_r, r_rand=r_r, f_rand=f_r
                ))
                print(f"  ✓ {style}/{fname}")

            except Exception as e:
                print(f"  ✗ {fname}: {e}")

    return pd.DataFrame(rows)


results = run_dataset_evaluation(AUDIO_ROOT, BEAT_CSV_DIR)
print(f"\nTotal files evaluated: {len(results)}")

## 9. Results — Summary Table

In [ ]:
summary = pd.DataFrame({
    'System':    ['Baseline', 'Improved', 'Random'],
    'Precision': [results['p_base'].mean(), results['p_impr'].mean(), results['p_rand'].mean()],
    'Recall':    [results['r_base'].mean(), results['r_impr'].mean(), results['r_rand'].mean()],
    'F-measure': [results['f_base'].mean(), results['f_impr'].mean(), results['f_rand'].mean()],
}).round(4)

print("=== Dataset-Wide Mean Scores ===")
print(summary.to_string(index=False))

## 10. Visualizations

### 10a. F-measure Distribution — Baseline vs Improved vs Random

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(results['f_base'], bins=20, alpha=0.55, color='steelblue', label='Baseline')
ax.hist(results['f_impr'], bins=20, alpha=0.55, color='crimson',   label='Improved')
ax.hist(results['f_rand'], bins=20, alpha=0.55, color='grey',      label='Random')

ax.axvline(results['f_base'].mean(), color='steelblue', linestyle='--', lw=2,
           label=f"Baseline mean = {results['f_base'].mean():.2f}")
ax.axvline(results['f_impr'].mean(), color='crimson',   linestyle='--', lw=2,
           label=f"Improved mean = {results['f_impr'].mean():.2f}")
ax.axvline(results['f_rand'].mean(), color='grey',      linestyle='--', lw=2,
           label=f"Random mean   = {results['f_rand'].mean():.2f}")

ax.set_xlabel('F-measure')
ax.set_ylabel('Number of files')
ax.set_title('F-measure Distribution across All Files')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 10b. Mean F-measure by Genre — Improved System

In [ ]:
genre_means = results.groupby('style')[['f_base', 'f_impr', 'f_rand']].mean()
genre_means.columns = ['Baseline', 'Improved', 'Random']

genre_means.plot(kind='bar', figsize=(12, 5), colormap='Set2')
plt.title('Mean F-measure by Genre')
plt.ylabel('F-measure')
plt.xlabel('Genre')
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

### 10c. Scatterplot — Baseline F vs Improved F (per file)

Points above the diagonal line = improved system wins on that file.  
Points below = baseline wins.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Colour by genre
genres = results['style'].unique()
cmap   = plt.cm.get_cmap('tab10', len(genres))
for idx, genre in enumerate(genres):
    sub = results[results['style'] == genre]
    ax.scatter(sub['f_base'], sub['f_impr'], alpha=0.7, s=25,
               color=cmap(idx), label=genre)

# Diagonal = equal performance
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Equal (y=x)')
ax.set_xlabel('Baseline F-measure')
ax.set_ylabel('Improved F-measure')
ax.set_title('Per-file F-measure: Baseline vs Improved\n(above diagonal = improved wins)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
plt.show()

n_improved = (results['f_impr'] > results['f_base']).sum()
n_baseline = (results['f_base'] > results['f_impr']).sum()
print(f"Improved wins on {n_improved} / {len(results)} files")
print(f"Baseline wins on {n_baseline} / {len(results)} files")

### 10d. Precision vs Recall — All Files

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, col_p, col_r, col_f, title, color in [
    (axes[0], 'p_base', 'r_base', 'f_base', 'Baseline',  'steelblue'),
    (axes[1], 'p_impr', 'r_impr', 'f_impr', 'Improved',  'crimson'),
]:
    sc = ax.scatter(results[col_p], results[col_r],
                    c=results[col_f], cmap='RdYlGn',
                    vmin=0, vmax=1, alpha=0.7, s=30)
    ax.set_xlabel('Precision')
    ax.set_ylabel('Recall')
    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.colorbar(sc, ax=ax, label='F-measure')

plt.suptitle('Precision vs Recall per File (colour = F-measure)', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Model Summary Table

In [ ]:
model_table = pd.DataFrame([
    {
        'Model':         'Baseline',
        'Feature':       'Onset strength (spectral flux)',
        'Novelty fn':    'Raw envelope',
        'Peak picking':  'Fixed global threshold',
        'hop_length':    512,
        'frame_length':  'N/A',
        'threshold':     0.3,
        'distance':      2,
    },
    {
        'Model':         'Improved',
        'Feature':       'RMS energy',
        'Novelty fn':    'Derivative + half-wave rectification',
        'Peak picking':  'Local maxima + threshold',
        'hop_length':    512,
        'frame_length':  1024,
        'threshold':     0.15,
        'distance':      'N/A',
    },
    {
        'Model':         'Random (naïve)',
        'Feature':       'None',
        'Novelty fn':    'None',
        'Peak picking':  'Uniform random times',
        'hop_length':    'N/A',
        'frame_length':  'N/A',
        'threshold':     'N/A',
        'distance':      'N/A',
    },
])

model_table.set_index('Model')